# MD-GRTN Phase 2 — 3-Period Input (Recent + Yesterday + 1-Week Ago)

**Upgraded from Phase 1 (MAE=15.170):**

| Component | Phase 1 | Phase 2 |
|---|---|---|
| Input sequences | 1 (recent only) | **3 (recent + yesterday + 1-week)** |
| BackNets | 1 | **3 (one per period)** |
| MAF heads | 1 | **3 → Concat+FC** |
| Dataset | single sliding window | **3 aligned windows** |
| num_layers | 5 | 5 (kept) |
| gru_layers | 3 | 3 (kept) |

**3 Period offsets (5-min steps):**
- `X_RecN`  → `t-12 : t`         (most recent 1 hr)
- `X_HourN` → `t-300 : t-288`    (yesterday same hour, offset=288 steps=24hr)
- `X_DayN`  → `t-2028 : t-2016`  (1 week ago same hour, offset=2016 steps=7 days)

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} ✓')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    # data
    data_path   = "/content/PEMS08.npz"
    num_nodes   = 170
    in_features = 3               # flow, occupancy, speed
    seq_len     = 12              # 12 steps × 5 min = 1 hr window
    pred_len    = 12
    feature_idx = 0               # predict flow

    # 3-period offsets in timesteps (5-min intervals)
    # X_RecN  : offset = 0    → t-12  to t         (recent 1 hr)
    # X_HourN : offset = 288  → t-300 to t-288     (yesterday same time)
    # X_DayN  : offset = 2016 → t-2028 to t-2016   (1 week ago same time)
    offset_rec  = 0
    offset_hour = 288    # 288 × 5min = 24 hours
    offset_week = 2016   # 2016 × 5min = 7 days

    # model — n_seqs=3 for 3-period MDAF
    d_model    = 64
    n_heads    = 4
    n_seqs     = 3       # ← 3 sequences now
    num_layers = 5
    gru_layers = 3
    dropout    = 0.1

    # training
    batch_size  = 32
    lr          = 1e-3
    epochs      = 100
    patience    = 15
    train_ratio = 0.6
    val_ratio   = 0.2

    # checkpoint
    ckpt_path = 'mdgrtn_phase2_ckpt.pt'
    best_path = 'mdgrtn_phase2_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config ready | n_seqs={cfg.n_seqs} | num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers}')
print(f'Offsets → recent={cfg.offset_rec} | yesterday={cfg.offset_hour} steps | week={cfg.offset_week} steps')
print(f'Device: {device}')

In [ ]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    print('Keys:', list(raw.keys()))
    data = raw['data'].astype(np.float32)   # (T, N, F)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')

    # Per-sensor z-score normalisation
    mean = data.mean(axis=0, keepdims=True)  # (1, N, F)
    std  = data.std(axis=0,  keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    for key in ('adjacency_matrix','adj_mx','adj'):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f'Adjacency from "{key}" shape={A_dist.shape}')
            break
    else:
        print('No adjacency → using identity')
        A_dist = np.eye(N, dtype=np.float32)

    deg    = A_dist.sum(axis=1, keepdims=True) + 1e-8
    A_dist = A_dist / deg
    return data_norm, mean, std, A_dist


class MultiPeriodDataset(Dataset):
    """
    Returns (x_rec, x_hour, x_week, y) per sample.

    Window layout (all seq_len=12 steps = 1 hr):
      x_rec  : data[t-12        : t]           recent
      x_hour : data[t-12-288    : t-288]       yesterday same time
      x_week : data[t-12-2016   : t-2016]      1 week ago same time
      y      : data[t           : t+12, :, fi] prediction target
    """
    def __init__(self, data, cfg, start, end):
        self.data  = torch.from_numpy(data)
        self.S     = cfg.seq_len
        self.P     = cfg.pred_len
        self.fi    = cfg.feature_idx
        self.oh    = cfg.offset_hour   # 288
        self.ow    = cfg.offset_week   # 2016
        # earliest valid t: need week-ago window available
        min_t      = cfg.offset_week + cfg.seq_len
        self.idx   = list(range(start + min_t, end - cfg.pred_len + 1))

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        t  = self.idx[i]
        S, P, fi = self.S, self.P, self.fi

        x_rec  = self.data[t-S       : t]                  # (S,N,F) recent
        x_hour = self.data[t-S-self.oh : t-self.oh]        # (S,N,F) yesterday
        x_week = self.data[t-S-self.ow : t-self.ow]        # (S,N,F) 1-week ago
        y      = self.data[t          : t+P, :, fi]        # (P,N)

        return x_rec, x_hour, x_week, y


def build_dataloaders(cfg):
    set_seed()
    data, mean, std, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(MultiPeriodDataset(data,cfg,0, t1), shuffle=True,  **kw)
    dl_va = DataLoader(MultiPeriodDataset(data,cfg,t1,t2), shuffle=False, **kw)
    dl_te = DataLoader(MultiPeriodDataset(data,cfg,t2,T),  shuffle=False, **kw)

    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    print(f'(Note: first {cfg.offset_week+cfg.seq_len} timesteps skipped for week-ago offset)')
    return dl_tr, dl_va, dl_te, mean, std, A_dist

print('Multi-period dataset ready (recent + yesterday + 1-week).')

In [ ]:
# ══════════════════════════════════════════════════
# MDAF with n_seqs=3
# 3 separate BackNets (one per period) + MAF fusion
# Architecture figure: BackNet1/2/3 → Linear×3 → DotProduct → Concat+FC
# ══════════════════════════════════════════════════

class BackNet(nn.Module):
    """MLP denoiser per period. Residual connection for stronger denoising."""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim),
        )
    def forward(self, x): return self.net(x)   # (B,S,N,F)


class MultiHeadAttentionFusion(nn.Module):
    """
    Per-period projection → dot-product attention → Concat → FC.
    n_seqs=3: one attention head per period, outputs concatenated.
    """
    def __init__(self, in_dim, d_model, n_heads, n_seqs=3):
        super().__init__()
        # Separate projection per period (paper: 3 Linear per period in figure)
        self.projs  = nn.ModuleList([nn.Linear(in_dim, d_model) for _ in range(n_seqs)])
        self.attn   = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # Concat(3 periods) → FC → d_model
        self.fc_out = nn.Linear(d_model * n_seqs, d_model)

    def forward(self, seqs):
        """seqs: list of n_seqs × (B,S,N,F) → (B,S,N,d_model)"""
        B, S, N, _ = seqs[0].shape
        projected = []
        for proj, seq in zip(self.projs, seqs):
            h = proj(seq.reshape(B*S, N, -1))     # (B*S, N, d_model)
            h, _ = self.attn(h, h, h)             # dot-product over N nodes
            projected.append(h.reshape(B, S, N, -1))
        # Concat all periods → FC (paper Eq.9)
        return self.fc_out(torch.cat(projected, dim=-1))  # (B,S,N,d)


class MDModule(nn.Module):
    """3 separate BackNets — one per period (paper: BackNet1, BackNet2, BackNet3)."""
    def __init__(self, in_features, d_model, n_seqs=3):
        super().__init__()
        self.backnets = nn.ModuleList([BackNet(in_features, d_model) for _ in range(n_seqs)])

    def forward(self, seqs):
        return [bn(s) for bn, s in zip(self.backnets, seqs)]


class MDAFModule(nn.Module):
    def __init__(self, in_features, d_model, n_heads, n_seqs=3):
        super().__init__()
        self.md  = MDModule(in_features, d_model, n_seqs)
        self.maf = MultiHeadAttentionFusion(in_features, d_model, n_heads, n_seqs)

    def forward(self, seqs):
        """seqs: list of 3 × (B,S,N,F) → X_F1: (B,S,N,d)"""
        cleaned = self.md(seqs)     # denoise each period
        return self.maf(cleaned)    # fuse all 3

print('MDAF defined (n_seqs=3: BackNet×3 + MAF).')

In [ ]:
class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
    def forward(self, x, A):
        return self.lin(torch.einsum('nm,bmd->bnd', A, x))


class GCN_GRU_Layer(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.gcn = GraphConv(in_dim, hidden_dim)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

    def forward(self, x_seq, A):
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            gcn_out = F.relu(self.gcn(x_seq[:,t], A))
            h       = self.gru(gcn_out.reshape(B*N,-1), h)
            outs.append(h.reshape(B,N,-1))
        return torch.stack(outs, dim=1)


class MGRCModule(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_nodes, num_layers=3):
        super().__init__()
        self.emb      = nn.Parameter(torch.randn(num_nodes, hidden_dim) * 0.01)
        self.adj_conv = nn.Conv2d(2, 1, kernel_size=1)
        dims = [in_dim] + [hidden_dim]*num_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i+1]) for i in range(num_layers)])

    def get_fused_adj(self, A_dist):
        A_dyna  = torch.softmax(self.emb @ self.emb.T, dim=-1)
        stacked = torch.stack([A_dist, A_dyna], dim=0).unsqueeze(0)
        A_F     = F.relu(self.adj_conv(stacked).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_dist):
        A_F = self.get_fused_adj(A_dist)
        for layer in self.layers:
            x = layer(x, A_F)
        return x

print('MGRC defined.')

In [ ]:
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.ReLU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.reshape(B*S, N, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model*4), nn.ReLU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print('Transformer layers defined (5 spatial + 5 temporal).')

In [ ]:
class MDGRTN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        P, S = cfg.pred_len, cfg.seq_len

        # MDAF with n_seqs=3 (3 BackNets + 3-period MAF)
        self.mdaf = MDAFModule(F, d, h, n_seqs=cfg.n_seqs)
        self.mgrc = MGRCModule(d, d, N, num_layers=cfg.gru_layers)

        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, N, d) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, S, 1, d) * 0.02)

        self.spatial_layers  = nn.ModuleList([SpatialTransformerLayer(d, h, dr) for _ in range(L)])
        self.temporal_layers = nn.ModuleList([TemporalTransformerLayer(d, h, dr) for _ in range(L)])

        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x_rec, x_hour, x_week, A_dist):
        """
        x_rec, x_hour, x_week : (B, S, N, F)
        A_dist                 : (N, N)
        returns                : (B, P, N)
        """
        # MDAF: 3 BackNets denoise → MAF fuses → X_F1
        X_F1 = self.mdaf([x_rec, x_hour, x_week])     # (B,S,N,d)

        # MGRC: dual adjacency → stacked GCN+GRU
        X_F2 = self.mgrc(X_F1, A_dist)                # (B,S,N,d)

        # STFormer: spatial + temporal transformers
        x = X_F2 + self.spatial_pos
        for layer in self.spatial_layers:
            x = layer(x)
        x = x + self.temporal_pos
        for layer in self.temporal_layers:
            x = layer(x)

        # Output
        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)  # (B,P,N)


set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  n_seqs={cfg.n_seqs} | num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers}')

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, dummy, dummy, adj)
print(f'Output shape: {out.shape}  ✓  (expected [2, {cfg.pred_len}, {cfg.num_nodes}])')

In [ ]:
def masked_mae(pred, true):
    return torch.abs(pred - true).mean()

def masked_rmse(pred, true):
    return ((pred - true)**2).mean().sqrt()

def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

def huber_loss(pred, true, delta=1.0):
    return F.huber_loss(pred, true, delta=delta)

print('Metrics defined.')

In [ ]:
# ── Mount Drive (uncomment if needed) ──
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

# Per-sensor denorm — shape (N,)
mean_flow = torch.from_numpy(mean_np[0,:,cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [0,:,cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

print(f'mean_flow: min={mean_flow.min():.2f} max={mean_flow.max():.2f}')
print(f'std_flow:  min={std_flow.min():.2f}  max={std_flow.max():.2f}')
print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')

In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_week, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_week = x_week.to(device)
        y      = y.to(device)

        pred = model(x_rec, x_hour, x_week, A_dist)
        loss = huber_loss(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_week, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_week = x_week.to(device)
        y      = y.to(device)

        pred   = model(x_rec, x_hour, x_week, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]

        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval functions defined (3-input forward pass).')

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    ckpt = {
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch'          : epoch,
        'best_mae'       : best_mae,
        'history'        : history,
        'seed'           : SEED,
        'n_seqs'         : cfg.n_seqs,
        'num_layers'     : cfg.num_layers,
        'gru_layers'     : cfg.gru_layers,
        'd_model'        : cfg.d_model,
    }
    torch.save(ckpt, cfg.ckpt_path)
    if drive_path:
        import shutil; shutil.copy(cfg.ckpt_path, drive_path)
        print(f'Saved to Drive ✓')
    else:
        print(f'Ckpt saved (ep {epoch} | best_mae={best_mae:.3f})', end='\r')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    if drive_path:
        import shutil; shutil.copy(drive_path, cfg.ckpt_path)
    if not os.path.exists(cfg.ckpt_path):
        print('No checkpoint — starting fresh.')
        return 1, float('inf'), {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
    ckpt = torch.load(cfg.ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    print(f'Resumed ep {ckpt["epoch"]} | best_mae={ckpt["best_mae"]:.3f}')
    return ckpt['epoch']+1, ckpt['best_mae'], ckpt['history']

print('Checkpoint utilities ready.')

In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6)

# ── To RESUME uncomment: ──
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_phase2_ckpt.pt')

DRIVE_CKPT   = None   # set to Drive path to auto-save
start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}

print(f'Phase 2 Training | 3 sequences | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Optimizer: AdamW | Schedule: CosineAnnealing | Loss: Huber')
print(f'Phase 1 result → MAE=15.170 | Baseline → MAE=13.114')
print('='*68)

for epoch in range(start_epoch, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f}  RMSE={val_rmse:.3f}  MAPE={val_mape:.2f}%{tag}')

    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', label='Huber Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(15.170, color='blue', ls=':', lw=1.5, label='Phase 1: 15.170')
axes[1].axhline(13.114, color='red',  ls='--', lw=1.5, label='Baseline: 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(24.548, color='blue', ls=':', lw=1.5, label='Phase 1: 24.548')
axes[2].axhline(22.623, color='red',  ls='--', lw=1.5, label='Baseline: 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_phase2.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_week, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_week = x_week.to(device)
        y      = y.to(device)

        pred   = model(x_rec, x_hour, x_week, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*60)
    print('  TEST RESULTS  —  averaged over all 12 steps')
    print('='*60)
    print(f'  MAE  : {mae:.3f}    Phase1: 15.170   Baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    Phase1: 24.548   Baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   Phase1:  8.513%  Baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*60)
    return mae, rmse, mape

paper_style_eval(model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_week, y in loader:
        x_rec  = x_rec.to(device)
        x_hour = x_hour.to(device)
        x_week = x_week.to(device)
        y      = y.to(device)
        pred   = model(x_rec, x_hour, x_week, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)